# 06 — Route choice: choice sets and path-size logit

Equilibrium assignment assumes drivers only ever take cost-minimal routes. **Route
choice models** instead generate a *set* of plausible routes per OD pair and split
demand among them with a discrete-choice model. AequilibraE implements:

- **BFSLE** (breadth-first search with link elimination) and **link penalisation**
  for choice-set generation — both extremely fast Cython implementations;
- **Path-Size Logit (PSL)** assignment, which corrects the IIA problem for
  overlapping routes.

We use the Coquimbo model, with utility = distance × θ.


In [1]:
from pathlib import Path
from tempfile import gettempdir
from uuid import uuid4

import numpy as np

from aequilibrae.utils.create_example import create_example

fldr = str(Path(gettempdir()) / uuid4().hex)
project = create_example(fldr, "coquimbo")

theta = 0.00011                       # utility per metre
nodes_of_interest = (71645, 74089, 77011, 79385)

project.network.build_graphs()
graph = project.network.graphs["c"]
graph.network = graph.network.assign(utility=graph.network.distance * theta)
graph.prepare_graph(np.array(nodes_of_interest))
graph.set_graph("utility")

C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: UserWarning: Found centroids not present in the graph!
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46  47  48  49  50  51  52  53  54
  55  56  57  58  59  60  61  62  63  64  65  66  67  68  69  70  71  72
  73  74  75  76  77  78  79  80  81  82  83  84  85  86  87  88  89  90
  91  92  93  94  95  96  97  98  99 100 101 102 103 104 105 106 107 108
 109 110 111 112 113 114 115 116 117 118 119 120 121 122 123 124 125 126
 127 128 129 130 131 132 133]
  build_compressed_graph(self, remove_dead_ends)
C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the i

C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: ChainedAssignmentError: A value is being set on a copy of a DataFrame or Series through chained assignment.
Such chained assignment never works to update the original DataFrame or Series, because the intermediate object on which we are setting values always behaves as a copy (due to Copy-on-Write).

Try using '.loc[row_indexer, col_indexer] = value' instead, to perform the assignment in a single step.

See the documentation for a more detailed explanation: https://pandas.pydata.org/pandas-docs/stable/user_guide/copy_on_write.html#chained-assignment
  build_compressed_graph(self, remove_dead_ends)
C:\Users\Riz\Desktop\AequilibraE\aequilibrae\aequilibrae\paths\graph.py:218: UserWarning: Found centroids not present in the graph!
[  1   2   3   4   5   6   7   8   9  10  11  12  13  14  15  16  17  18
  19  20  21  22  23  24  25  26  27  28  29  30  31  32  33  34  35  36
  37  38  39  40  41  42  43  44  45  46 

In [2]:
# A small synthetic demand matrix between our nodes of interest
from aequilibrae.matrix import AequilibraeMatrix

mat = AequilibraeMatrix()
mat.create_empty(zones=graph.num_zones, matrix_names=["demand"], memory_only=True)
mat.index = graph.centroids[:]
mat.matrices[:, :, 0] = np.full((graph.num_zones, graph.num_zones), 10.0)
mat.computational_view()

In [3]:
from aequilibrae.paths import RouteChoice

rc = RouteChoice(graph)
rc.add_demand(mat)

# Always bound the generation: at most 5 routes per OD pair here
rc.set_choice_set_generation("bfsle", max_routes=5)
rc.default_parameters

{'generic': {'seed': 0,
  'max_routes': 0,
  'max_depth': 0,
  'max_misses': 100,
  'penalty': 1.01,
  'cutoff_prob': 0.0,
  'beta': 1.0,
  'store_results': True},
 'link-penalisation': {},
 'bfsle': {'penalty': 1.0}}

In [4]:
# Generate a choice set for one OD pair and assign its demand with PSL
results = rc.execute_single(77011, 74089, demand=1.0)
print(f"{len(results)} routes found between 77011 and 74089")
results[0][:12]  # link ids of the first route

5 routes found between 77011 and 74089


(np.int64(-24222),
 np.int64(30332),
 np.int64(30333),
 np.int64(-10435),
 np.int64(30068),
 np.int64(30069),
 np.int64(14198),
 np.int64(14199),
 np.int64(31161),
 np.int64(30928),
 np.int64(-31622),
 np.int64(24112))

In [5]:
# Offline map helper ---------------------------------------------------------
# Interactive maps with no server extensions, no labextensions beyond the
# ipywidgets manager, and no CDN: lonboard renders WebGL maps whose frontend
# JavaScript ships from the kernel through the ipywidgets channel.
#
# Backends (AEQ_MAP_BACKEND environment variable):
#   lonboard (default) - interactive WebGL maps (pip install lonboard anywidget)
#   static             - matplotlib rendering, works absolutely anywhere
#
# The declarative symbology below (field()/constant() chains) is self-contained
# and renders identically on both backends.
import os

import matplotlib.colors
import matplotlib.pyplot as _plt
import numpy as np


# --- declarative symbology --------------------------------------------------
class _Mapping:
    def __init__(self, field, scheme, params):
        self.field, self.scheme, self.params = field, scheme, params

    def encoding(self, *targets):
        return {"field": self.field, "scheme": self.scheme,
                "params": self.params, "encodings": list(targets)}


class _Field:
    def __init__(self, name):
        self.name = name

    def colormap(self, name="viridis", *, domain=None, reverse=False, n_shades=9):
        return _Mapping(self.name, "colormap",
                        {"name": name, "domain": domain, "reverse": reverse})

    def scalar(self, *, domain, output_range):
        return _Mapping(self.name, "scalar",
                        {"domain": list(domain), "range": list(output_range)})

    def categorical(self, name="tab10"):
        return _Mapping(self.name, "categorical", {"name": name})


class _Constant:
    def __init__(self, value):
        self.value = value

    def encoding(self, *targets):
        scheme = "constant_num" if isinstance(self.value, (int, float)) else "constant_color"
        return {"field": None, "scheme": scheme,
                "params": {"value": self.value}, "encodings": list(targets)}


def field(name):
    """Style by a data column: .colormap() / .scalar() / .categorical()."""
    return _Field(name)


def constant(value):
    """A fixed colour (hex/name) or number, e.g. constant("#dc2626")."""
    return _Constant(value)


def _rgba255(c, alpha=1.0):
    r, g, b, a = matplotlib.colors.to_rgba(c, alpha)
    return [int(r * 255), int(g * 255), int(b * 255), int(a * 255)]


def _style_arrays(symbology, gdf):
    """symbology -> per-row uint8 RGBA arrays and float width arrays."""
    n = len(gdf)
    out = {"stroke": None, "width": None, "fill": None}
    if not symbology:
        return out
    mappings = [m for group in symbology for m in (group if isinstance(group, list) else [group])]
    for m in mappings:
        scheme, params, fld, encs = m["scheme"], m["params"], m["field"], m["encodings"]
        arr = wid = None
        if scheme == "constant_color":
            arr = np.tile(_rgba255(params["value"]), (n, 1)).astype(np.uint8)
        elif scheme == "colormap":
            cmap = _plt.get_cmap(params["name"])
            if params.get("reverse"):
                cmap = cmap.reversed()
            dom = params.get("domain") or [float(gdf[fld].min()), float(gdf[fld].max())]
            vals = gdf[fld].to_numpy(dtype=float)
            t = np.clip((vals - dom[0]) / max(dom[1] - dom[0], 1e-12), 0, 1)
            rgba = cmap(t)
            arr = (rgba * 255).astype(np.uint8)
        elif scheme == "categorical":
            cmap = _plt.get_cmap(params["name"])
            uniq = list(dict.fromkeys(gdf[fld].dropna()))
            idx = {v: i for i, v in enumerate(uniq)}
            arr = np.array([_rgba255(cmap(idx.get(v, 0) % cmap.N)) for v in gdf[fld]], dtype=np.uint8)
        elif scheme == "constant_num":
            wid = np.full(n, float(params["value"]))
        elif scheme == "scalar":
            d, r = params["domain"], params["range"]
            vals = gdf[fld].to_numpy(dtype=float)
            t = np.clip((vals - d[0]) / max(d[1] - d[0], 1e-12), 0, 1)
            wid = r[0] + t * (r[1] - r[0])
        if arr is not None:
            if any("stroke" in e for e in encs):
                out["stroke"] = arr
            if any("fill" in e for e in encs):
                out["fill"] = arr
        if wid is not None and any("width" in e for e in encs):
            out["width"] = wid
    return out


# --- the map document -------------------------------------------------------
class MapDoc:
    """Collects styled layers; displays via lonboard (WebGL) or matplotlib."""

    def __init__(self):
        self.items = []  # (gdf, name, arrays, opacity)

    def add(self, gdf, name, symbology, opacity):
        g = gdf.reset_index(drop=True).explode(index_parts=False).reset_index(drop=True)
        self.items.append((g, name, _style_arrays(symbology, g), opacity))

    def _lonboard_map(self):
        from lonboard import Map, PathLayer, PolygonLayer, ScatterplotLayer
        layers = []
        for g, name, st, op in self.items:
            if not len(g):
                continue
            geom = g.geometry.geom_type.iloc[0]
            base = g[["geometry"]]
            if "LineString" in geom:
                kw = {"width_units": "pixels", "width_min_pixels": 1.0, "opacity": op}
                if st["stroke"] is not None:
                    kw["get_color"] = st["stroke"]
                if st["width"] is not None:
                    kw["get_width"] = st["width"]
                layers.append(PathLayer.from_geopandas(base, **kw))
            elif "Polygon" in geom:
                kw = {"opacity": op * 0.6, "stroked": False}
                if st["fill"] is not None:
                    kw["get_fill_color"] = st["fill"]
                layers.append(PolygonLayer.from_geopandas(base, **kw))
            else:
                kw = {"radius_min_pixels": 5, "opacity": op}
                fill = st["fill"] if st["fill"] is not None else st["stroke"]
                if fill is not None:
                    kw["get_fill_color"] = fill
                layers.append(ScatterplotLayer.from_geopandas(base, **kw))
        return Map(layers=layers, basemap=None)

    def _static_figure(self):
        fig, ax = _plt.subplots(figsize=(9, 7))
        ax.set_facecolor("#eef1f4")
        for g, name, st, op in self.items:
            if not len(g):
                continue
            geom = g.geometry.geom_type.iloc[0]
            if "LineString" in geom:
                colors = st["stroke"] / 255 if st["stroke"] is not None else "#1d4ed8"
                widths = st["width"] if st["width"] is not None else 1.0
                g.plot(ax=ax, color=colors, linewidth=widths, alpha=op)
            elif "Polygon" in geom:
                colors = st["fill"] / 255 if st["fill"] is not None else "#cbd5e1"
                g.plot(ax=ax, color=colors, alpha=op * 0.6)
            else:
                fill = st["fill"] if st["fill"] is not None else st["stroke"]
                g.plot(ax=ax, color=(fill / 255 if fill is not None else "#dc2626"),
                       markersize=25, alpha=op)
        ax.set_aspect(1.4)
        ax.set_xticks([]), ax.set_yticks([])
        _plt.tight_layout()
        _plt.close(fig)
        return fig

    def _ipython_display_(self):
        from IPython.display import display
        be = os.environ.get("AEQ_MAP_BACKEND", "lonboard").strip().lower()
        display(self._static_figure() if be == "static" else self._lonboard_map())


def new_map(gdf_for_extent=None, zoom=12):
    """Create a map document (extent/zoom args kept for API compatibility;
    lonboard auto-fits to its layers)."""
    return MapDoc()


def add_gdf(doc, gdf, name, symbology=None, **kwargs):
    """Add a GeoDataFrame to the map as a styled layer."""
    doc.add(gdf, name, symbology, kwargs.get("opacity", 1.0))
    return name


def merge_lines(gdf, tol=0.01):
    """Collapse many lines into a single MultiLineString feature — backdrop
    layers do not need per-feature identity, and one merged feature is a
    fraction of the size and draw cost."""
    import geopandas as _gpd
    from shapely.geometry import MultiLineString
    parts = []
    for geom in gdf.geometry.simplify(tol):
        if geom is None or geom.is_empty:
            continue
        parts.extend(geom.geoms if geom.geom_type == "MultiLineString" else [geom])
    return _gpd.GeoDataFrame({"links": [len(parts)]}, geometry=[MultiLineString(parts)], crs=gdf.crs)


In [6]:
# field()/constant() symbology builders come from the map helper cell

links = project.network.links.data
palette = ["#dc2626", "#2563eb", "#16a34a", "#d97706", "#7c3aed"]

route_links = links[links.link_id.isin({l for route in results for l in route})]
doc = new_map(route_links, zoom=13)
add_gdf(doc, links, "network", opacity=0.4, symbology=[[constant("#94a3b8").encoding("stroke")]])
for i, route in enumerate(results):
    add_gdf(doc, links[links.link_id.isin(route)], f"route {i + 1}",
            symbology=[[constant(palette[i % len(palette)]).encoding("stroke")]])
doc

[interactive offline map - run the notebook to display]

## Batch assignment

`prepare()` + `execute(perform_assignment=True)` runs generation and PSL assignment
for **every** OD pair in the demand matrix, giving link-level loads.


In [7]:
rc.prepare()
rc.execute(perform_assignment=True)

loads = rc.get_load_results()
loads.sort_values("demand_tot", ascending=False).head()

,demand_ab,demand_ba,demand_tot
link_id,,,
20964,27.121247,33.905817,61.027064
20963,27.121247,33.905817,61.027064
20965,27.121247,33.905817,61.027064
20962,27.121247,33.905817,61.027064
29899,27.121247,33.905817,61.027064


In [8]:
# field()/constant() symbology builders come from the map helper cell

loaded = links.merge(loads.reset_index(), on="link_id")
loaded = loaded[loaded["demand_tot"] > 0]

doc2 = new_map(loaded, zoom=12)
vmax = float(loaded["demand_tot"].max())
add_gdf(doc2, loaded[["link_id", "demand_tot", "geometry"]], "route-choice flows",
        symbology=[[field("demand_tot").colormap("viridis", domain=(0.0, vmax)).encoding("stroke"),
                field("demand_tot").scalar(domain=(0.0, vmax), output_range=(0.8, 6.5)).encoding("stroke-width")]])
doc2

[interactive offline map - run the notebook to display]

In [9]:
project.close()

INFO:aequilibrae:Closed project on C:\Users\Riz\AppData\Local\Temp\b2930483c3d447eeaa4147427649a87f


---
**Next:** [07 — Public transport](07_public_transport.ipynb): importing GTFS and building
a transit model.
